# Qwen PRM — System Prompt Effect on Scores

Examine how different system prompts affect `Qwen2.5-Math-PRM-7B`
per-step scores. Uses the inline `score_qwen_prm` function (no
wrapper) so a single model load can be reused across all prompts.

Two examples scored under each system prompt:
1. Flamingo problem (all steps correct)
2. Algebra correct vs. wrong trajectory

Env: py311 / transformers 4.57, fp16 on V100.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc
import sys
sys.path.append("..")

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

from notebook_utils import gpu_mem_used_gb, print_step_scores
from utils.configs import system_prompt as genconfig_system_prompt

base_dir = "/groups/chichengz/tnn/datasets"
qwen_prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

## System prompts

Four variants to compare:
- **qwen_sys_prompt** — Qwen model-card prompt (boxed answer
  instruction)
- **short** — minimal instruction, no boxed-answer requirement
- **custom_sys_prompt** — `GenConfig.system_prompt` from
  `utils/configs.py`
- **empty** — no system prompt at all

In [2]:
system_prompts = {
    "empty": "",
    "qwen_sys_prompt": (
        "Please reason step by step, and put your final answer "
        "within \\boxed{}."
    ),
    "custom_sys_prompt": genconfig_system_prompt,
    "short": "Solve the problem step by step.",
}

## Scoring function

In [3]:
def score_qwen_prm(
    model,
    tokenizer,
    problem: str,
    steps: list[str],
    system: str,
    step_separator: str = "<extra_0>",
    print_conversation: bool = False,
) -> list[float]:
    assistant = step_separator.join(steps) + step_separator
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": assistant},
    ]
    conversation = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    if print_conversation:
        print("===== Formatted PRM input =====")
        print(conversation)
        print("===============================")
    input_ids = tokenizer.encode(
        conversation, return_tensors="pt"
    ).to(model.device)

    sep_ids = tokenizer.encode(
        step_separator, add_special_tokens=False
    )
    if len(sep_ids) != 1:
        raise ValueError(
            f"Expected one separator token, got {sep_ids}"
        )
    sep_positions = (
        input_ids[0] == sep_ids[0]
    ).nonzero(as_tuple=True)[0]
    if sep_positions.numel() != len(steps):
        raise RuntimeError(
            f"Expected {len(steps)} separators, "
            f"found {sep_positions.numel()}"
        )

    with torch.no_grad():
        logits = model(input_ids=input_ids, use_cache=False)[0]

    probs = F.softmax(logits, dim=-1)
    return probs[0, sep_positions, 1].detach().cpu().float().tolist()

## Load the PRM

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    qwen_prm_dir, trust_remote_code=True
)
model = AutoModel.from_pretrained(
    qwen_prm_dir,
    device_map="cuda:0",
    dtype=torch.float16,
    trust_remote_code=True,
).eval()

print(f"dtype          : {next(model.parameters()).dtype}")
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

dtype          : torch.float16
GPU memory used: 27.73 GB


## Example 1 — flamingo problem

All four steps are correct. We expect high scores throughout;
the question is whether the system prompt shifts the absolute
values or changes the relative ranking.

In [5]:
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [6]:
for name, system in system_prompts.items():
    scores = score_qwen_prm(
        model, tokenizer, problem, reasoning_steps, system
    )
    print(f"=== system_prompt: {name!r} ===")
    print_step_scores(reasoning_steps, scores)
    print()

=== system_prompt: 'empty' ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1458
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9668
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9990
To find the difference, subtract the number of white flaming...

=== system_prompt: 'qwen_sys_prompt' ===
Step 1: P(correct) = 0.9995
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1580
On Saturday, they take back one third of the flamingos. Sinc...
Step 3: P(correct) = 0.9741
On Sunday, the neighbors add another 18 pink plastic flaming...
Step 4: P(correct) = 0.9995
To find the difference, subtract the number of white flaming...

=== system_prompt: 'custom_sys_prompt' ===
Step 1: P(correct) = 0.9990
To find out how many more pink plastic flamingos were out th...
Step 2: P(correct) = 0.1219
On Saturday, they take back

## Example 2 — correct vs. wrong trajectory

Equation `3x + 5 = 17`. The wrong trajectory divides by 2 instead
of 3 at step 2. We check whether the system prompt affects how
sharply the PRM flags the error.

In [7]:
algebra_problem = "If 3x + 5 = 17, what is x?"

correct_steps = [
    "We need solve the equation 3x + 5 = 17.",
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 3 gives x = 4.",
    "Therefore, the answer is (\\boxed{4}).",
]

wrong_steps = [
    "Subtracting 5 from both sides gives 3x = 12.",
    "Dividing both sides by 2 gives x = 6.",
    "Therefore, the final answer is \\boxed{6}.",
]

In [8]:
# Show the formatted conversation for qwen_sys_prompt,
# custom_sys_prompt, and empty with the correct algebra
# trajectory.
for name in ("empty", "qwen_sys_prompt", "custom_sys_prompt"):
    print(f"===== sys_prompt: {name!r} =====")
    _ = score_qwen_prm(
        model, tokenizer,
        algebra_problem, correct_steps,
        system=system_prompts[name],
        print_conversation=True,
    )
    print()

===== sys_prompt: 'empty' =====
===== Formatted PRM input =====
<|im_start|>system
<|im_end|>
<|im_start|>user
If 3x + 5 = 17, what is x?<|im_end|>
<|im_start|>assistant
We need solve the equation 3x + 5 = 17.<extra_0>Subtracting 5 from both sides gives 3x = 12.<extra_0>Dividing both sides by 3 gives x = 4.<extra_0>Therefore, the answer is (\boxed{4}).<extra_0><|im_end|><|endoftext|>

===== sys_prompt: 'qwen_sys_prompt' =====
===== Formatted PRM input =====
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
If 3x + 5 = 17, what is x?<|im_end|>
<|im_start|>assistant
We need solve the equation 3x + 5 = 17.<extra_0>Subtracting 5 from both sides gives 3x = 12.<extra_0>Dividing both sides by 3 gives x = 4.<extra_0>Therefore, the answer is (\boxed{4}).<extra_0><|im_end|><|endoftext|>

===== sys_prompt: 'custom_sys_prompt' =====
===== Formatted PRM input =====
<|im_start|>system
Solve the following math problem efficiently and 

In [9]:
for name, system in system_prompts.items():
    correct_scores = score_qwen_prm(
        model, tokenizer, algebra_problem, correct_steps, system
    )
    wrong_scores = score_qwen_prm(
        model, tokenizer, algebra_problem, wrong_steps, system
    )
    print(f"=== system_prompt: {name!r} ===")
    print("--- Correct trajectory ---")
    print_step_scores(correct_steps, correct_scores)
    print("--- Wrong trajectory (step 2 divides by 2) ---")
    print_step_scores(wrong_steps, wrong_scores)
    print()

=== system_prompt: 'empty' ===
--- Correct trajectory ---
Step 1: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).
--- Wrong trajectory (step 2 divides by 2) ---
Step 1: P(correct) = 0.9897
Subtracting 5 from both sides gives 3x = 12.
Step 2: P(correct) = 0.0120
Dividing both sides by 2 gives x = 6.
Step 3: P(correct) = 0.4285
Therefore, the final answer is \boxed{6}.

=== system_prompt: 'qwen_sys_prompt' ===
--- Correct trajectory ---
Step 1: P(correct) = 0.9995
We need solve the equation 3x + 5 = 17.
Step 2: P(correct) = 0.9995
Subtracting 5 from both sides gives 3x = 12.
Step 3: P(correct) = 1.0000
Dividing both sides by 3 gives x = 4.
Step 4: P(correct) = 1.0000
Therefore, the answer is (\boxed{4}).
--- Wrong trajectory (step 2 divides by 2) ---
Step 1: P(correct) = 0.9

## Cleanup

In [10]:
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

GPU memory used: 27.43 GB
